In [13]:
import pandas as pd
import numpy as np
import os
from backtester import backtest

DATA_DIR = "./data"


In [ ]:
class Model:
    def __init__(self, ops):
        self.bars_seen_train         = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
        self.bars_unseen_train       = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
        self.bars_seen_public_test   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
        self.bars_seen_private_test  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

        self.headlines_seen_train        = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_train.parquet"), engine="fastparquet")
        self.headlines_unseen_train      = pd.read_parquet(os.path.join(DATA_DIR, "headlines_unseen_train.parquet"), engine="fastparquet")
        self.headlines_seen_public_test  = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_public_test.parquet"), engine="fastparquet")
        self.headlines_seen_private_test = pd.read_parquet(os.path.join(DATA_DIR, "headlines_seen_private_test.parquet"), engine="fastparquet")
        self.ops = ops
        print("bars_seen_train:",         self.bars_seen_train.shape)
        print("bars_unseen_train:",       self.bars_unseen_train.shape)
        print("bars_seen_public_test:",   self.bars_seen_public_test.shape)
        print("bars_seen_private_test:",  self.bars_seen_private_test.shape)
        print("headlines_seen_train:",        self.headlines_seen_train.shape)
        print("headlines_unseen_train:",      self.headlines_unseen_train.shape)
        print("headlines_seen_public_test:",  self.headlines_seen_public_test.shape)
        print("headlines_seen_private_test:", self.headlines_seen_private_test.shape)


    def fit(self):
        lags = self.ops["lag"]
        
        # Check autocorrelation and distribution of the log returns 
        def autocorrelation(df, lags = 1):
            autocorrs = []
            for col in df.columns:
                series = df[col].dropna()
                if len(series) > lags:
                    autocorrs.append(series.autocorr(lag=lags))
            return autocorrs
        
        open_price_df = self.bars_seen_train[["session", "bar_ix", "open"]]
        open_price_pivot = open_price_df.pivot(index="bar_ix", columns="session", values="open")


        log_ret_open = np.log(open_price_pivot / open_price_pivot.shift(1)).dropna()

        autocorr_lag1 = autocorrelation(df=log_ret_open, lags=lags)

        # print 5 most and least autocorrelated sessions
        autocorr_df = pd.DataFrame({
            "session": log_ret_open.columns,
            "autocorr_lag1": autocorr_lag1
        })  
        autocorr_df = autocorr_df.dropna().sort_values("autocorr_lag1", ascending=False).reset_index(drop=True)
        print("Top 5 most autocorrelated sessions (lag 1):")
        print(autocorr_df.head(5))
        print("\nTop 5 least autocorrelated sessions (lag 1):")
        print(autocorr_df.tail(5))

    def predict(self):
        bottom_n = self.ops["bottom_n"]
        top_n = self.ops["top_n"]
        lag = self.ops["lag"]
        
        def autocorr_strategy(bars_df):
            """
            Sort sessions by autocorr(lag):
            - Top top_n   (most positive): apply Kelly bet  clip(ac * r_lag / σ, -3, 3)
            - Bottom bottom_n (most negative): same Kelly bet
            - Rest: volume = 1  (flat long)
            """
            open_pivot = bars_df.pivot(index="bar_ix", columns="session", values="open")
            log_ret    = np.log(open_pivot / open_pivot.shift(1)).dropna()

            # Compute autocorr and signal for every session
            rows = []
            for sid in log_ret.columns:
                r     = log_ret[sid].dropna().values
                sigma = r.std()
                if sigma < 1e-8 or len(r) <= lag:
                    rows.append({"session": sid, "autocorr": 0.0, "signal": 0.0, "sigma": sigma})
                    continue
                ac     = pd.Series(r).autocorr(lag=lag)
                r_lag  = r[lag - 1]
                rows.append({"session": sid, "autocorr": ac, "signal": ac * r_lag, "sigma": sigma})

            df = pd.DataFrame(rows).sort_values("autocorr", ascending=False).reset_index(drop=True)

            top_idx    = df.index[:top_n]
            bottom_idx = df.index[-bottom_n:]
            active_idx = top_idx.union(bottom_idx)

            volumes = []
            for idx, row in df.iterrows():
                if idx in active_idx:
                    vol = float(np.clip(row["signal"] / row["sigma"], -3.0, 3.0)) if row["sigma"] > 1e-8 else 1.0
                else:
                    vol = 1.0
                volumes.append(vol)

            df["target_position"] = volumes
            return df[["session", "target_position", "autocorr"]]
        
        # autocorr is a strat, rest is glue code to run the predictions and produce a csv

        print("Predicting train set")
        train_seen_only = self.bars_seen_train[self.bars_seen_train["bar_ix"] <= 49].copy()
        train_preds     = autocorr_strategy(self.bars_seen_train)
        train_results   = backtest(train_preds.rename(columns={"target_position": "volume"})[["session", "volume"]])

        print("Predicting test set")
        # ── Apply to both test sets ───────────────────────────────────────────────
        print("Predicting public")
        public_preds  = autocorr_strategy(self.bars_seen_public_test)
        print("Predicting private")
        private_preds = autocorr_strategy(self.bars_seen_private_test)

        public_preds[["session", "target_position"]].to_csv("predictions_public_autocorr25.csv",  index=False)
        private_preds[["session", "target_position"]].to_csv("predictions_private_autocorr25.csv", index=False)
        print("Saved predictions_public_autocorr25.csv  and  predictions_private_autocorr25.csv")



In [ ]:
ops = {
    "lag": 25, "top_n" : 300, "bottom_n" : 200
}

model = Model(ops)

bars_seen_train: (50000, 6)
bars_unseen_train: (50000, 6)
bars_seen_public_test: (500000, 6)
bars_seen_private_test: (500000, 6)
headlines_seen_train: (9740, 3)
headlines_unseen_train: (7631, 3)
headlines_seen_public_test: (99308, 3)
headlines_seen_private_test: (99148, 3)
Top 5 most autocorrelated sessions (lag 1):
   session  autocorr_lag1
0      853       0.600031
1      573       0.583092
2      297       0.551723
3      623       0.541164
4      292       0.520713

Top 5 least autocorrelated sessions (lag 1):
     session  autocorr_lag1
995      516      -0.558607
996      574      -0.570594
997       10      -0.625016
998      961      -0.636076
999      336      -0.662526


In [20]:
model.predict()

Predicting train set
  Sessions evaluated : 1000
  Sharpe ratio       : +2.0978
  Mean PnL           : +0.001925
  Std  PnL           : 0.014681
  Win rate           : 51.30%
Predicting test set
Predicting public
Predicting private
Saved predictions_public_autocorr25.csv  and  predictions_private_autocorr25.csv
